# PCA False-Color Visualization

This notebook maps the first three principal components of each pixel's
embedding vector to R/G/B, producing a false-color image that reveals
spatial structure learned by OlmoEarth.

**Prerequisites:** Run `PYTHONPATH=src python -m olmoearth_embeddings_tutorial.pca.compute` first to download
embeddings and Sentinel-2 RGB for Flevoland.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt

# Allow imports from the tutorial src/ directory
sys.path.insert(0, str(Path.cwd().parent / "src"))

from olmoearth_embeddings_tutorial.common.constants import (
    EMBEDDINGS_FILENAME,
    S2_RGB_FILENAME,
)
from olmoearth_embeddings_tutorial.pca.analyze import compute_pca_rgb, make_pca_figure

In [ ]:
EMBED_PATH = Path("../data/flevoland") / EMBEDDINGS_FILENAME
RGB_PATH = Path("../data/flevoland") / S2_RGB_FILENAME

result = compute_pca_rgb(EMBED_PATH)

In [ ]:
print(
    f"Variance explained: "
    f"PC1={result.variance_pct[0]:.1f}%  "
    f"PC2={result.variance_pct[1]:.1f}%  "
    f"PC3={result.variance_pct[2]:.1f}%  "
    f"(total {result.variance_pct.sum():.1f}%)"
)

In [ ]:
import numpy as np
import rasterio

s2_rgb = None
if RGB_PATH.exists():
    with rasterio.open(RGB_PATH) as ds:
        rgb = ds.read([1, 2, 3]).astype(np.float32)
    rgb = np.moveaxis(rgb, 0, -1)
    lo, hi = np.nanpercentile(rgb, [2, 98])
    s2_rgb = np.clip((rgb - lo) / max(float(hi - lo), 1e-6), 0, 1)
    s2_rgb[np.isnan(s2_rgb).any(axis=-1)] = 0.15

fig = make_pca_figure(result, s2_rgb)
plt.show()